# Кривое из прямого, свет из уравнений

**Как выбрать поверхность, чтобы конструкция выполняла свою задачу?**

**Версия преподавателя** · Практикум 10 · 90 минут. Вы спроектируете каркас из прямых стержней и проверите, можно ли заставить сферическое зеркало работать лучше, перемещая приёмник. В обоих случаях вы будете читать каноническое уравнение через плоские сечения и отличать математическую поверхность от её рабочего участка.

**Версия 0.5.0.** Статический режим проверен локально.

## Два прогноза перед началом

Может ли целый прямой стержень лежать на гладкой поверхности с талией? Если два зеркала почти совпадают около вершины, можно ли просто подобрать положение приёмника, чтобы оба собирали параллельные лучи в точку? Запишите два прогноза с кратким обоснованием; в конце вернитесь к ним.

**Ваши прогнозы:** …

## Как работать

Первые 5 минут — прогнозы и запуск. Затем 25 минут на каркас, 50 минут на зеркала и 10 минут на выходной билет. Работайте в парах: один предлагает решение, другой ищет, чем его проверить. Во второй истории поменяйтесь ролями.

В Colab загрузите скачанный файл через **Файл → Загрузить блокнот**. Запускайте ячейки сверху вниз. Начальный служебный блок можно оставить свёрнутым. Значение «не заполнено» не мешает запуску главы. Если интерактивная фигура не появилась, задайте interactive=False в ячейке опыта и запустите её снова. В статическом режиме меняйте параметры обычным запуском ячейки.

Проверка публикации в служебном блоке необязательна и выключена по умолчанию. Включайте её отдельно, если хотите сравнить версии. Ваши ответы остаются в текущем блокноте.

In [ ]:
# @title Условия задач
FRAME_RADIUS_M = 2.0
FRAME_HEIGHT_M = 4.0
FRAME_WAIST_M = 1.0
SPHERE_RADIUS_M = 4.0
APERTURE_RADIUS_M = 1.5
TARGET_FOCUS_M = 2.0

NOTEBOOK_ID = "week-10-quadrics"
NOTEBOOK_VERSION = "0.5.0"
NOTEBOOK_RELEASE_DATE = None
NOTEBOOK_PATH = "weeks/week-10-quadrics/notebooks/straight-structures-focused-light-teacher.ipynb"

In [ ]:
# @title Геометрические вычисления
try:
    import numpy as np
except ImportError as error:
    raise RuntimeError("Для запуска нужны NumPy и Matplotlib. Используйте стандартную среду Colab или установите недостающий пакет в своём ядре.") from error

def frame_state(offset_deg, section_z, sweep_deg):
    if not (0 <= offset_deg <= 180 and abs(section_z) <= FRAME_HEIGHT_M/2
            and 0 <= sweep_deg <= 360):
        raise ValueError("Проверьте диапазоны угла, высоты и угла вращения.")
    R, H = FRAME_RADIUS_M, FRAME_HEIGHT_M
    half = np.deg2rad(offset_deg)/2
    a = 0.0 if offset_deg == 180 else R*np.cos(half)
    k = 2*R*np.sin(half)/H
    z = np.linspace(-H/2, H/2, 49)
    radius = np.hypot(a, k*z)
    phi = np.linspace(0, np.deg2rad(sweep_deg), 65)
    x = radius[:, None]*np.cos(phi)
    y = radius[:, None]*np.sin(phi)
    theta = np.linspace(0, 2*np.pi, 97)
    rings = [(r*np.cos(theta), r*np.sin(theta), np.full_like(theta, h))
             for r, h in [(R,-H/2),(R,H/2),(np.hypot(a,k*section_z),section_z)]]
    phi0 = np.linspace(0, 2*np.pi, 24, endpoint=False)
    phi1 = phi0 + 2*half
    lower = np.column_stack((R*np.cos(phi0), R*np.sin(phi0), np.full(24,-H/2)))
    upper = np.column_stack((R*np.cos(phi1), R*np.sin(phi1), np.full(24,H/2)))
    return dict(x=x, y=y, z=z, radius=radius, rings=rings, lower=lower,
                upper=upper, waist=a, slope=k, section_z=section_z)

_MIRROR_NAMES = {"sphere": "Сферическое зеркало", "paraboloid": "Параболическое зеркало"}

def mirror_geometry(kind, xy, aperture=APERTURE_RADIUS_M):
    """Return points and unit normals, with one point per row."""
    xy = np.asarray(xy, dtype=float)
    if (kind not in _MIRROR_NAMES or not np.isfinite(aperture)
            or not 0 < aperture < SPHERE_RADIUS_M):
        raise ValueError("Проверьте тип зеркала и радиус апертуры.")
    if xy.ndim != 2 or xy.shape[1] != 2 or not np.isfinite(xy).all():
        raise ValueError("Нужен массив конечных пар (x, y).")
    r = np.hypot(xy[:,0], xy[:,1])
    if np.any(r > aperture + 16*np.spacing(aperture)):
        raise ValueError("Точка находится вне рабочей части зеркала.")
    r2 = r*r
    if kind == "sphere":
        R = SPHERE_RADIUS_M
        h = np.sqrt(R*R-r2)
        z = r2/(R+h)  # Avoid cancellation close to the vertex.
        nz = -h
    else:
        f = TARGET_FOCUS_M
        z = r2/(4*f)
        nz = np.full_like(z,-2*f)
    normals = np.column_stack((xy,nz))
    normals /= np.linalg.norm(normals,axis=1,keepdims=True)
    return np.column_stack((xy,z)), normals

def reflect_rays(direction, unit_normals):
    d = np.asarray(direction, dtype=float)
    n = np.asarray(unit_normals, dtype=float)
    if (d.shape != (3,) or n.ndim != 2 or n.shape[1] != 3
            or not np.isclose(np.linalg.norm(d),1)
            or not np.allclose(np.linalg.norm(n,axis=1),1)):
        raise ValueError("Направление и нормали должны быть единичными 3D-векторами.")
    return d - 2*(n @ d)[:,None]*n

def detector_hits(points, directions, detector_z):
    """Return hit coordinates and a mask. Invalid rays keep their row as NaN."""
    p, v = np.asarray(points, float), np.asarray(directions, float)
    if (p.ndim != 2 or p.shape[1] != 3 or p.shape != v.shape
            or not np.isfinite(p).all() or not np.isfinite(v).all()
            or not np.isfinite(detector_z)):
        raise ValueError("Проверьте координаты лучей и высоту детектора.")
    tau = np.full(len(p),np.nan)
    np.divide(detector_z-p[:,2], v[:,2], out=tau,
              where=np.abs(v[:,2]) > 1e-12)
    valid = np.isfinite(tau) & (tau >= 0)
    q = np.full_like(p,np.nan)
    q[valid] = p[valid]+tau[valid,None]*v[valid]
    return q, valid

_DETECTOR_BOUNDS = (1.5, 2.5)

def aperture_bundle(kind, aperture=APERTURE_RADIUS_M, *, validation=False):
    if not 0 < aperture <= APERTURE_RADIUS_M:
        raise ValueError("Используйте радиус апертуры от 0 до 1,5 м (0 исключён).")
    rings, sectors = (256, 24) if validation else (64, 12)
    r = aperture*np.sqrt((np.arange(rings)+.5)/rings)
    phase = (np.arange(rings)[:,None] % 2)/2 + (.25 if validation else 0)
    angle = 2*np.pi*(np.arange(sectors)[None,:]+phase)/sectors
    xy = np.column_stack(((r[:,None]*np.cos(angle)).ravel(),
                          (r[:,None]*np.sin(angle)).ravel()))
    points, normals = mirror_geometry(kind,xy,aperture)
    return points, reflect_rays((0,0,-1),normals)

def all_hit_xy(bundle, detector_z):
    if not _DETECTOR_BOUNDS[0] <= detector_z <= _DETECTOR_BOUNDS[1]:
        raise ValueError("Высота детектора должна лежать между 1,5 и 2,5 м.")
    hits, valid = detector_hits(*bundle, detector_z)
    if not valid.all():
        raise ValueError(f"Пропущено лучей: {np.count_nonzero(~valid)}. Оценка недоступна.")
    return hits[:,:2]

def detector_mse(bundle, detector_z):
    xy = all_hit_xy(bundle, detector_z)
    return float(np.mean(np.sum(xy**2,axis=1)))

def detector_search(bundle):
    lo, hi = _DETECTOR_BOUNDS
    coarse = np.linspace(lo,hi,21)
    loss = [detector_mse(bundle,z) for z in coarse]
    i = int(np.argmin(loss))
    fine = np.unique(np.r_[coarse[i], np.linspace(
        coarse[max(0,i-1)],coarse[min(len(coarse)-1,i+1)],101)])
    best = min(fine,key=lambda z: detector_mse(bundle,z))
    return coarse[i], best

In [ ]:
# @title Визуализация каркаса
try:
    import matplotlib.pyplot as plt
except ImportError as error:
    raise RuntimeError("Для запуска нужны NumPy и Matplotlib. Используйте стандартную среду Colab или установите недостающий пакет в своём ядре.") from error

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    go = None

_FRAME_BLUE, _FRAME_ORANGE = "#175986", "#a34a10"

def _frame_title(data):
    x, y, _ = data["rings"][2]
    return f'z = {data["section_z"]:.2f} м · радиус = {np.hypot(x[0], y[0]):.3f} м'

def _frame_static(data, view):
    fig = plt.figure(figsize=(11, 4), layout="constrained", facecolor="white")
    ax = fig.add_subplot(131, projection="3d")
    profile, cut = fig.add_subplot(132), fig.add_subplot(133)
    z, r = data["z"], data["radius"]
    X, Y = data["x"], data["y"]
    if view in ("surface", "both") and np.any(Y):
        ax.plot_surface(X, Y, np.broadcast_to(z[:,None], X.shape),
                        color=_FRAME_BLUE, alpha=.24, linewidth=0)
    elif view in ("surface", "both"):
        ax.plot(X[:,0], Y[:,0], z, color=_FRAME_BLUE)
    if view in ("members", "both"):
        for i, (a,b) in enumerate(zip(data["lower"], data["upper"])):
            points = np.stack((a,b)).T
            ax.plot(*points, color=_FRAME_ORANGE if i == 0 else _FRAME_BLUE,
                    lw=2.5 if i == 0 else .85)
            if i == 0:
                ax.scatter(*np.stack((a,(a+b)/2,b)).T, color=_FRAME_ORANGE, s=16)
    for i, ring in enumerate(data["rings"]):
        ax.plot(*ring, color=_FRAME_ORANGE if i == 2 else "#334155",
                ls="--" if i == 2 else "-", lw=2)
    ax.set(xlabel="x, м", ylabel="y, м", zlabel="z, м", title="Каркас в пространстве",
           xlim=(-2.3,2.3), ylim=(-2.3,2.3), zlim=(-2.3,2.3))
    ax.set_box_aspect((1,1,1))
    ax.set_proj_type("ortho")
    profile.plot(r,z,-r,z,color=_FRAME_BLUE)
    profile.axhline(data["section_z"], color=_FRAME_ORANGE, ls="--")
    profile.set(xlabel="x, м",ylabel="z, м",title="Осевое сечение: y = 0")
    x,y,_ = data["rings"][2]
    cut.plot(x,y,color=_FRAME_ORANGE,ls="--", marker="o" if not np.any(x) else None)
    cut.set(xlabel="x, м",ylabel="y, м",title=_frame_title(data))
    for panel in (profile,cut):
        panel.set(xlim=(-2.3,2.3),ylim=(-2.3,2.3),aspect="equal")
        panel.grid(alpha=.2)
    return fig

def _frame_plotly(angle, height, sweep, view, control):
    fig = make_subplots(rows=1,cols=3, specs=[[{"type":"scene"},{"type":"xy"},{"type":"xy"}]],
                       subplot_titles=("Каркас в пространстве","Осевое сечение: y = 0",
                                       "Горизонтальное сечение"))
    def traces(d):
        X,Y,z,r = d["x"],d["y"],d["z"],d["radius"]
        blue, orange = _FRAME_BLUE,_FRAME_ORANGE
        x,y,_ = d["rings"][2]
        a,b = d["lower"],d["upper"]
        rods = np.insert(np.stack((a,b),axis=1), 2, np.nan, axis=1).reshape(-1,3).T
        member = np.stack((a[0], (a[0]+b[0])/2, b[0])).T
        line = lambda color, width, dash="solid": dict(color=color,width=width,dash=dash)
        return [
            go.Surface(x=X,y=Y,z=np.broadcast_to(z[:,None],X.shape),opacity=.28,
                       colorscale=[[0,blue],[1,blue]],showscale=False,hoverinfo="skip",
                       visible=view in ("surface","both")),
            go.Scatter3d(x=X[:,0],y=Y[:,0],z=z,mode="lines", line=line(blue,4),
                         visible=view in ("surface","both"),name="Образующая кривая"),
            go.Scatter3d(x=rods[0],y=rods[1],z=rods[2],mode="lines",line=line(blue,3),
                         visible=view in ("members","both"),name="Стержни"),
            go.Scatter3d(x=member[0],y=member[1],z=member[2],mode="lines+markers",
                         line=line(orange,6),marker=dict(size=4,color=orange,symbol="diamond"),
                         visible=view in ("members","both"),name="Концы и середина стержня"),
            *[go.Scatter3d(x=u,y=v,z=w,mode="lines+markers" if i == 2 and not np.any(u) else "lines",
                           line=line(orange if i==2 else "#334155",4,"dash" if i==2 else "solid"),
                           marker=dict(size=4,color=orange),
                           name="Горизонтальный срез" if i==2 else "Кольцо",showlegend=i==2)
              for i,(u,v,w) in enumerate(d["rings"])],
            go.Scatter(x=np.r_[r,np.nan,-r],y=np.r_[z,np.nan,z],mode="lines",
                       line=line(blue,2),showlegend=False),
            go.Scatter(x=[-2.3,2.3],y=[d["section_z"]]*2,mode="lines",
                       line=line(orange,2,"dash"),showlegend=False),
            go.Scatter(x=x,y=y,mode="markers" if not np.any(x) else "lines",
                       line=line(orange,3,"dash"),marker=dict(color=orange,size=9),showlegend=False),
        ]
    current = {"height":height,"angle":angle,"sweep":sweep}[control]
    grid, label = {
        "height":(np.linspace(-2,2,17),"Высота z, м: "),
        "angle":(np.linspace(0,180,19),"Сдвиг δ, °: "),
        "sweep":(np.linspace(0,360,13),"Угол вращения, °: "),
    }[control]
    states = sorted(set([*grid,current]))
    def state(value):
        params = dict(angle=angle,height=height,sweep=sweep)
        params[control] = value
        return frame_state(params["angle"],params["height"],params["sweep"])
    initial = state(current)
    for i, trace in enumerate(traces(initial)):
        fig.add_trace(trace,row=1,col=1 if i<7 else (2 if i<9 else 3))
    steps, frames = [], []
    for i,value in enumerate(states):
        d = state(value)
        frames.append(go.Frame(name=str(i),data=traces(d),traces=list(range(10)),
                               layout=dict(title_text=_frame_title(d))))
        steps.append(dict(label=f"{value:g}",method="animate",
                          args=[[str(i)],dict(mode="immediate",frame=dict(duration=0,redraw=True),
                                              transition=dict(duration=0))]))
    fig.frames = frames
    fig.update_layout(template="plotly_white", height=510, title_text=_frame_title(initial),
                      uirevision="frame", margin=dict(l=18,r=12,t=85,b=75),
                      legend=dict(orientation="h",y=-.25),
                      sliders=[dict(active=states.index(current),steps=steps,currentvalue=dict(prefix=label))])
    fig.update_scenes(aspectmode="data",camera_projection_type="orthographic",
                      xaxis=dict(title="x, м",range=[-2.3,2.3]),
                      yaxis=dict(title="y, м",range=[-2.3,2.3]),
                      zaxis=dict(title="z, м",range=[-2.3,2.3]))
    fig.update_xaxes(title_text="x, м",range=[-2.3,2.3])
    fig.update_yaxes(title_text="z, м",range=[-2.3,2.3],scaleanchor="x",row=1,col=2)
    fig.update_yaxes(title_text="y, м",range=[-2.3,2.3],scaleanchor="x2",row=1,col=3)
    return fig

def show_frame(offset_deg, section_z=0, sweep_deg=360, view="both",
               control="height", interactive=True):
    data = frame_state(offset_deg, section_z, sweep_deg)
    print(_frame_title(data), f'· радиус талии = {data["waist"]:.3f} м')
    if interactive and go is not None:
        try:
            _frame_plotly(offset_deg, section_z, sweep_deg, view, control).show()
            return
        except (ImportError, RuntimeError, ValueError):
            print("Интерактивный режим недоступен; показываем статический рисунок.")
    _frame_static(data, view)
    plt.show()

def _mirror_panels(with_rays):
    """Data shared by the interactive and static axial diagrams."""
    rho = APERTURE_RADIUS_M
    x = np.linspace(-rho,rho,181)
    axial = lambda x: np.column_stack((x,np.zeros_like(x)))
    panels = []
    for kind in _MIRROR_NAMES:
        profile, _ = mirror_geometry(kind, axial(x))
        lines = [(profile[:,0],profile[:,2],"Профиль",_FRAME_BLUE,False)]
        if with_rays:
            p,n = mirror_geometry(kind, axial(np.linspace(-rho,rho,11)))
            v = reflect_rays((0,0,-1),n)
            # Clip outgoing segments at a common height for comparison.
            q, valid = detector_hits(p,v,2.35)
            if not valid.all():
                raise ValueError("Луч не пересёк верхнюю границу рисунка.")
            for name,start,end,color,dashed in [
                ("Падающие лучи",np.column_stack((p[:,0],p[:,1],np.full(len(p),.72))),p,"#64748b",True),
                ("Отражённые лучи",p,q,_FRAME_ORANGE,False),
            ]:
                segments = np.concatenate(
                    (start[:,None,:],end[:,None,:],np.full((len(p),1,3),np.nan)),axis=1
                ).reshape(-1,3)
                lines.append((segments[:,0],segments[:,2],name,color,dashed))
        panels.append(lines)
    return panels

def _mirror_static(panels, with_rays):
    fig, axs = plt.subplots(1,2,figsize=(10,5 if with_rays else 3),
                            layout="constrained",facecolor="white")
    for ax,title,lines in zip(axs,_MIRROR_NAMES.values(),panels):
        for x,z,name,color,dashed in lines:
            ax.plot(x,z,label=name,color=color,ls="--" if dashed else "-",
                    lw=2.3 if name=="Профиль" else 1)
        ax.axvline(0,color="#94a3b8",lw=.8)
        if with_rays:
            ax.annotate("",xy=(1.5,.49),xytext=(1.5,.70),
                        arrowprops=dict(arrowstyle="->",color="#64748b"))
        ax.set(title=title,xlabel="x, м",ylabel="z, м",aspect="equal",
               xlim=(-1.65,1.65),ylim=(-.08,2.43 if with_rays else .4))
        ax.grid(alpha=.2)
    axs[0].legend(loc="upper left",fontsize="small")
    return fig

def _mirror_plotly(panels, with_rays):
    fig = make_subplots(rows=1,cols=2,subplot_titles=list(_MIRROR_NAMES.values()))
    for col,lines in enumerate(panels,1):
        for x,z,name,color,dashed in lines:
            fig.add_trace(go.Scatter(x=x,y=z,mode="lines",name=name,
                legendgroup=name,showlegend=col==1,
                line=dict(color=color,dash="dash" if dashed else "solid",
                          width=3 if name=="Профиль" else 1)),row=1,col=col)
        fig.update_xaxes(title_text="x, м",range=[-1.65,1.65],row=1,col=col)
        fig.update_yaxes(title_text="z, м",range=[-.08,2.43 if with_rays else .4],
                         scaleanchor="x" if col==1 else "x2",constrain="domain",
                         row=1,col=col)
        fig.add_vline(x=0,line_color="#94a3b8",line_width=1,row=1,col=col)
        if with_rays:
            fig.add_annotation(x=1.5,y=.49,ax=0,ay=-20,text="",showarrow=True,
                               arrowcolor="#64748b",row=1,col=col)
    fig.update_layout(template="plotly_white",height=485 if with_rays else 275,
                      legend=dict(orientation="h",y=-.16),margin=dict(t=45,b=65))
    return fig

def show_mirrors(with_rays=False, interactive=True):
    panels = _mirror_panels(with_rays)
    if interactive and go is not None:
        try:
            _mirror_plotly(panels,with_rays).show()
            return
        except (ImportError, RuntimeError, ValueError):
            print("Интерактивный режим недоступен; показываем статический рисунок.")
    _mirror_static(panels,with_rays)
    plt.show()

def _detector_view_data(kind, bundle, height):
    x = np.linspace(-APERTURE_RADIUS_M,APERTURE_RADIUS_M,11)
    points, normals = mirror_geometry(kind,np.column_stack((x,np.zeros_like(x))))
    direction = reflect_rays((0,0,-1),normals)
    hits, valid = detector_hits(points,direction,height)
    if not valid.all():
        raise ValueError(f"Пропущено лучей на схеме: {np.count_nonzero(~valid)}.")
    segments = np.concatenate(
        (points[:,None,:],hits[:,None,:],np.full((len(x),1,3),np.nan)),axis=1
    ).reshape(-1,3)
    xy = all_hit_xy(bundle,height)
    return segments[:,0],segments[:,2],xy,100*np.sqrt(detector_mse(bundle,height))

def show_detector(kind="sphere", detector_z=2.2, interactive=True):
    bundle = aperture_bundle(kind)
    grid = np.linspace(*_DETECTOR_BOUNDS,51)
    errors = [100*np.sqrt(detector_mse(bundle,z)) for z in grid]
    profile = _mirror_panels(False)[list(_MIRROR_NAMES).index(kind)][0]
    ray_x,ray_z,xy,rms = _detector_view_data(kind,bundle,detector_z)
    lim = max(.12, 105*max(np.abs(all_hit_xy(bundle,z)).max() for z in _DETECTOR_BOUNDS))
    print(f"Высота {detector_z:.3f} м · RMS {rms:.4g} см · лучей {len(xy)}")
    if interactive and go is not None:
        try:
            fig = make_subplots(rows=1,cols=3,
                subplot_titles=("Осевое сечение","Пятно на детекторе","Размер пятна"))
            fig.add_trace(go.Scatter(x=profile[0],y=profile[1],mode="lines",name="Зеркало",
                                    line=dict(color=_FRAME_BLUE)),row=1,col=1)
            fig.add_trace(go.Scatter(x=grid,y=errors,mode="lines",name="RMS",
                                    line=dict(color=_FRAME_BLUE)),row=1,col=3)
            def moving(z):
                x,y,hits,r = _detector_view_data(kind,bundle,z)
                return [
                    go.Scatter(x=x,y=y,mode="lines",line=dict(color=_FRAME_ORANGE,width=1)),
                    go.Scatter(x=[-1.6,1.6],y=[z,z],mode="lines",
                               line=dict(color="#334155",dash="dash")),
                    go.Scatter(x=100*hits[:,0],y=100*hits[:,1],mode="markers",
                               marker=dict(size=3,color=_FRAME_ORANGE,opacity=.5)),
                    go.Scatter(x=[z],y=[r],mode="markers",
                               marker=dict(color=_FRAME_ORANGE,size=8,symbol="diamond")),
                ]
            for trace,col in zip(moving(detector_z),(1,1,2,3)):
                fig.add_trace(trace,row=1,col=col)
            states = sorted(set([*grid,detector_z]))
            fig.frames = [go.Frame(name=str(i),data=moving(z),traces=[2,3,4,5])
                          for i,z in enumerate(states)]
            fig.update_layout(template="plotly_white",showlegend=False,height=405,
                sliders=[dict(active=states.index(detector_z),currentvalue=dict(prefix="Высота, м: "),
                    steps=[dict(label=f"{z:.3f}",method="animate",
                        args=[[str(i)],dict(mode="immediate",frame=dict(duration=0,redraw=False),
                                             transition=dict(duration=0))])
                           for i,z in enumerate(states)])])
            for col,xlabel,ylabel,xlim,ylim in _detector_axes(lim,errors):
                fig.update_xaxes(title_text=xlabel,range=xlim,row=1,col=col)
                fig.update_yaxes(title_text=ylabel,range=ylim,row=1,col=col)
            fig.update_yaxes(scaleanchor="x",constrain="domain",row=1,col=1)
            fig.update_yaxes(scaleanchor="x2",constrain="domain",row=1,col=2)
            fig.show()
            return
        except (ImportError,RuntimeError,ValueError):
            print("Интерактивный режим недоступен. Показываем статический рисунок.")
    fig,axes = plt.subplots(1,3,figsize=(11,3.4),layout="constrained")
    axes[0].plot(profile[0],profile[1],color=_FRAME_BLUE,lw=2)
    axes[0].plot(ray_x,ray_z,color=_FRAME_ORANGE,lw=1)
    axes[0].axhline(detector_z,color="#334155",ls="--")
    axes[1].scatter(*(100*xy).T,s=5,color=_FRAME_ORANGE,alpha=.55)
    axes[2].plot(grid,errors,color=_FRAME_BLUE)
    axes[2].scatter(detector_z,rms,marker="D",color=_FRAME_ORANGE,zorder=3)
    for (col,xlabel,ylabel,xlim,ylim),title in zip(_detector_axes(lim,errors),
            ("Осевое сечение","Пятно на детекторе","Размер пятна")):
        ax = axes[col-1]
        ax.set(title=title,xlabel=xlabel,ylabel=ylabel,xlim=xlim,ylim=ylim)
        ax.grid(alpha=.2)
    for ax in axes[:2]:
        ax.set_aspect("equal")
    plt.show()

def _detector_axes(lim,errors):
    return [(1,"x, м","z, м",[-1.65,1.65],[-.08,2.6]),
            (2,"x, см","y, см",[-lim,lim],[-lim,lim]),
            (3,"Высота, м","RMS, см",list(_DETECTOR_BOUNDS),[0,max(.01,1.08*max(errors))])]

def show_paraboloid_section(turn_deg=180):
    if not 0 <= turn_deg <= 360:
        raise ValueError("Введите угол от 0 до 360 градусов.")
    radius = np.linspace(0,APERTURE_RADIUS_M,32)
    angle = np.linspace(0,np.deg2rad(turn_deg),49)
    r,phi = np.meshgrid(radius,angle)
    fig = plt.figure(figsize=(9,3.4),layout="constrained")
    ax = fig.add_subplot(121,projection="3d")
    ax.plot_surface(r*np.cos(phi),r*np.sin(phi),r*r/(4*TARGET_FOCUS_M),
                    color=_FRAME_BLUE,alpha=.55)
    ax.plot(radius,np.zeros_like(radius),radius**2/(4*TARGET_FOCUS_M),color=_FRAME_ORANGE)
    ax.set(xlabel="x, м",ylabel="y, м",zlabel="z, м",title=f"Поворот профиля на {turn_deg}°",
           xlim=(-1.6,1.6),ylim=(-1.6,1.6),zlim=(0,.31))
    ax.set_box_aspect((3.2,3.2,.31))
    ax.set_zticks([0, .28])
    ax.tick_params(axis="z", labelsize=8, pad=1)
    cut = fig.add_subplot(122)
    section_r = APERTURE_RADIUS_M/np.sqrt(2)
    cut.plot(section_r*np.cos(angle),section_r*np.sin(angle),color=_FRAME_ORANGE)
    cut.set(title="Сечение на половине глубины",xlabel="x, м",ylabel="y, м",
            xlim=(-1.6,1.6),ylim=(-1.6,1.6),aspect="equal")
    cut.grid(alpha=.2)
    plt.show()

In [ ]:
# @title Служебная проверка черновика ответа
import math

def report(values):
    if any(value is None for value in values):
        print("○ НЕ ЗАПОЛНЕНО")
    elif any(type(value) not in (int, float) or not math.isfinite(value) for value in values):
        print("△ НУЖНО ИСПРАВИТЬ: введите конечные числа.")
    else:
        print("○ ВВОД ПРИНЯТ: сравните с самопроверкой в тексте.")

def report_frame(offset, length, residual):
    values = (offset, length, residual)
    if any(v is None for v in values) or any(
        type(v) not in (int,float) or not math.isfinite(v) for v in values
    ):
        report(values)
        return
    if not 0 <= offset <= 180:
        print("△ НУЖНО ИСПРАВИТЬ: угол задаётся в градусах, от 0 до 180.")
        return
    d = frame_state(offset, 0, 360)
    measured_length = np.linalg.norm(d["upper"][0]-d["lower"][0])
    if abs(d["waist"]-FRAME_WAIST_M) > .005:
        print("△ НУЖНО ИСПРАВИТЬ: с этим углом талия не соответствует чертежу.")
    elif abs(length-measured_length) > .01:
        print("△ НУЖНО ИСПРАВИТЬ: проверьте длину между концами стержня.")
    elif abs(residual) > .005:
        print("△ НУЖНО ИСПРАВИТЬ: проверьте подстановку внутренней точки.")
    else:
        print("✓ УСПЕХ: угол, длина и невязка согласуются с проектом.")
 
def valid_detector_height(value):
    return (isinstance(value,(int,float,np.integer,np.floating))
            and not isinstance(value,(bool,np.bool_))
            and np.isfinite(value)
            and _DETECTOR_BOUNDS[0] <= value <= _DETECTOR_BOUNDS[1])

def report_detector_loss(height, answer, bundle):
    if not valid_detector_height(height):
        print("△ НУЖНО ИСПРАВИТЬ: укажите высоту от 1,5 до 2,5 м.")
    elif answer is None:
        print("○ НЕ ЗАПОЛНЕНО")
    elif (isinstance(answer,(int,float,np.integer,np.floating))
          and not isinstance(answer,(bool,np.bool_))
          and np.isclose(answer,detector_mse(bundle,height),rtol=.005,atol=1e-12)):
        print("✓ УСПЕХ: значение J согласуется с попаданиями.")
    else:
        print("△ НУЖНО ИСПРАВИТЬ: проверьте квадраты координат, среднее и единицы.")

In [ ]:
import platform
from importlib.metadata import PackageNotFoundError, version

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "нет"

# Быстрая проба вычислений и построения в статическом режиме.
probe = frame_state(0,0,0)
figure, axis = plt.subplots()
axis.plot(probe["z"],probe["radius"])
plt.close(figure)
plotly_ready = False
if go is not None:
    try:
        go.Figure(go.Scatter(x=[0,1], y=[0,1])).to_json()
        plotly_ready = True
    except (ImportError, RuntimeError, TypeError, ValueError):
        go = None
print("Доступен интерактивный режим (при поддержке браузером)." if plotly_ready
      else "Доступен статический режим. Параметры меняются в ячейках.")
print(f"Python {platform.python_version()} · NumPy {installed_version('numpy')} · "
      f"Matplotlib {installed_version('matplotlib')} · Plotly {installed_version('plotly')}")

In [ ]:
# @title Проверка опубликованной версии
CHECK_FOR_UPDATE = False

def _version_parts(value):
    import re
    if not isinstance(value,str) or not re.fullmatch(r"[0-9]{1,4}(\.[0-9]{1,4}){2}",value):
        raise ValueError("Неверный формат версии")
    return tuple(map(int,value.split(".")))

def check_notebook_update(fetch=None):
    import json
    from datetime import date
    from urllib.request import urlopen

    repo = "elena-kornaeva/AGLA"
    manifest = f"https://raw.githubusercontent.com/{repo}/main/weeks/week-10-quadrics/notebooks/straight-structures-focused-light.version.json"
    limit = 4096
    if fetch is None:
        def fetch():
            with urlopen(manifest,timeout=3) as response:
                return response.read(limit+1)
    try:
        raw = fetch()
        if not isinstance(raw,bytes) or len(raw) > limit:
            raise ValueError("Неверный размер манифеста")
        info = json.loads(raw)
        if not isinstance(info,dict) or info.get("id") != NOTEBOOK_ID:
            raise ValueError("Неверный идентификатор")
        if info.get("path") != "weeks/week-10-quadrics/notebooks/straight-structures-focused-light.ipynb":
            raise ValueError("Неверный путь к блокноту")
        published = _version_parts(info["version"])
        date.fromisoformat(info["released"])
        current = _version_parts(NOTEBOOK_VERSION)
    except (OSError,ValueError,TypeError,KeyError):
        print("Не удалось проверить опубликованную версию. Можно продолжить работу.")
        return
    if published > current:
        message = "Доступна новая версия"
    elif published == current:
        message = "Версии совпадают"
    else:
        message = "Локальная версия новее опубликованной"
    print(f"{message}. Текущая: {NOTEBOOK_VERSION}. Опубликованная: {info['version']}.")
    print(f"https://colab.research.google.com/github/{repo}/blob/main/{NOTEBOOK_PATH}")

if CHECK_FOR_UPDATE:
    check_notebook_update()

## Карточка инструментов

Квадрика — множество решений уравнения \(F(x,y,z)=0\), где \(F\) — вещественный многочлен полной степени два. Чтобы получить сечение плоскостью \(z=h\), подставьте \(h\) вместо \(z\). Уравнение сечения может описывать кривую, точку или пустое множество. На поверхности вращения расстояние до оси \(z\) равно \(r=\sqrt{x^2+y^2}\).

Отрезок между точками \(A\) и \(B\) состоит из точек \(p(t)=(1-t)A+tB\), \(0\le t\le1\). Расстояния измеряем в метрах; угловые вычисления в Python выполняем в радианах. См. [страницу недели](../README.md) и [соглашения курса](../../../docs/notation.md) в репозитории.

## 1. Каркас: выберите форму · 5–15 мин

Два горизонтальных кольца радиуса 2 м находятся на расстоянии 4 м друг от друга. Посередине требуется талия радиуса 1 м. Боковая поверхность должна плавно соединять кольца. Сила и устойчивость конструкции здесь не рассчитываются.

| Кандидат | Уравнение в выбранной системе |
|---|---|
| Цилиндр | \(r^2=a^2\) |
| Эллипсоид вращения | \(r^2/a^2+z^2/c^2=1\) |
| Конус | \(r^2=k^2z^2\) |
| Однополостный гиперболоид | \(r^2/a^2-z^2/c^2=1\) |
| Двуполостный гиперболоид | \(z^2/c^2-r^2/a^2=1\) |
| Параболоид вращения | \(r^2=k(z-z_0)\), \(k>0\) |

Положите середину между кольцами на \(z=0\). Выберите уравнение и исключите остальные по поведению горизонтальных сечений. Найдите параметры и вычислите радиусы на \(z=0,1,2\). Затем нарисуйте сечение \(y=0\): какую кривую нужно вращать вокруг оси \(z\), чтобы получить рабочий участок?

**Подсказка после попытки:** подставьте \(z=2\) и \(r=2\).  
**Самопроверка:** уравнение возвращает размеры обоих колец и талии. Промежуточный радиус лежит строго между ними.

**Ваш выбор, расчёт и эскиз:** …

**После прогноза — опыт с моделью.** Здесь пробный сдвиг, поэтому размеры талии могут отличаться от заказа. Откройте поверхность вращением профиля; для опыта с высотой измените control на "height". Осевое и горизонтальное сечения показывают полную пробную поверхность, даже при неполном повороте.

In [ ]:
show_frame(offset_deg=75, section_z=1, sweep_deg=180, view="surface", control="sweep")

## 2. Каркас: спроектируйте стержень · 15–25 мин

Соедините точки нижнего и верхнего колец с одинаковым угловым сдвигом \(\delta\) для каждой пары. Для одного стержня возьмите \(A=(R,0,-H/2)\) и \(B=(R\cos\delta,R\sin\delta,H/2)\), где \(R\) — радиус колец, \(H\) — расстояние между ними и \(0\le\delta\le\pi\).

1. Найдите расстояние середины от оси. Какой сдвиг даёт нужную талию?
2. Найдите длину стержня по его горизонтальной и вертикальной составляющим.
3. Выберите внутренние значения \(t\), например \(1/4\) и \(3/4\). Подставьте \(p(t)\) в своё неявное уравнение и запишите невязку: левая часть минус правая.
4. Почему такой тест не заменяет доказательство для всех \(t\)?

**Подсказка после попытки:** для середины используйте \(1+\cos\delta=2\cos^2(\delta/2)\). Для общего случая подставьте \(z=H(t-1/2)\) и раскройте квадрат расстояния до оси.

**Самопроверка:** внутренняя точка стержня удовлетворяет уравнению с точностью округления; результат для середины совпадает с заданной талией. Для строгой проверки невязка как многочлен по \(t\) должна обращаться в ноль тождественно.

In [ ]:
# Введите угол в градусах, длину в метрах и невязку для t = 1/4.
offset_degrees = None
member_length_m = None
quarter_residual = None
report_frame(offset_degrees, member_length_m, quarter_residual)

**Проверьте проект.** Введите найденный угол выше и повторите запуск этой ячейки. Сравните радиусы при разных высотах. Параметр view принимает "members", "surface" или "both".

In [ ]:
if type(offset_degrees) in (int, float) and 0 <= offset_degrees <= 180:
    show_frame(offset_degrees, control="height")
else:
    print("Сначала введите допустимый угол в ячейке проекта.")

## 3. Каркас: поверните до предела · 25–30 мин

Нарисуйте отрезки при \(\delta=0^\circ\) и \(180^\circ\). Как меняется сечение посередине? Опишите всю поверхность для обоих пределов. Пользуйтесь координатами концов и промежуточных точек: параметры обычного гиперболоида в предельном случае могут стать непригодными.

**Самопроверка:** в одном из пределов сечение талии остаётся окружностью радиуса \(R\); в другом все середины совпадают. Объясните, как это видно без рисунка.

**Вызов:** сравните стержни для сдвигов \(+\delta\) и \(-\delta\) по их длинам и принадлежности одной поверхности.

**После эскиза — сравнение.** Откройте крайние положения. Пунктир показывает горизонтальное сечение; ромбы в интерактивной фигуре отмечают концы и середину одного стержня. Сдвиг задаёт новый проект с другой длиной стержней.

In [ ]:
show_frame(offset_deg=45, control="angle")

## 4. Зеркала: одинаковый ли результат? · 30–40 мин

Круглые зеркала имеют общую вершину в начале координат и общую ось \(z\). Их край расположен на расстоянии 1,5 м от оси. Рассмотрим нижнюю часть сферы \(x^2+y^2+(z-R_s)^2=R_s^2\), \(R_s=4\) м, и параболоид \(x^2+y^2=4fz\), \(f=2\) м. Оба открыты вверх. Модель описывает зеркальное отражение и не учитывает дифракцию и затенение приёмником.

**Сначала только профили.** Оба осевых сечения показаны в одном физическом масштабе; сравните зазор между вершиной и краем.

In [ ]:
show_mirrors(with_rays=False)

Нарисуйте осевые сечения. Сравните луч у края и луч около оси: какой из них способен показать различие между зеркалами? До вычислений запишите прогноз.

**Карточка отражения.** Для падающего единичного направления \(d=(0,0,-1)^T\) и единичной нормали \(\hat n\) отражённое направление равно \(v=d-2(d\cdot\hat n)\hat n\). Для сферы берите нормаль \((x,y,z-R_s)^T\), для параболоида — \((x,y,-2f)^T\). Разделите нормаль на её длину.

**Задание:** на сфере возьмите \(x=1\) м, \(y=0\), найдите \(z\) на нижней чаше и вычислите \(v\). **Самопроверка:** длина \(v\) равна единице; луч идёт вверх и в сторону оси. Замена знака нормали не меняет результат.

**Прогноз и расчёт одного луча:** …

**После расчёта — откройте лучи.** Пунктир показывает падающие лучи, стрелка у края — направление падения, сплошные оранжевые линии — отражение. Сравните, где крайние и близкие к оси лучи пересекают ось в каждой модели. Обновите свой прогноз одним предложением. На рисунке показано 11 лучей в осевой плоскости, а не выборка для оценки качества всей апертуры.

In [ ]:
show_mirrors(with_rays=True)

## 5. Приёмник: найдите лучшую настройку · 40–55 мин

Отражение происходит в точке \(p_i\), направление после отражения — \(v_i\). На плоскости \(z=s\) точка попадания равна
\[
q_i(s)=p_i+\frac{s-p_{iz}}{v_{iz}}v_i.
\]
Формула относится к попаданию вперёд по лучу: \(v_{iz}\ne0\), а множитель при \(v_i\) неотрицателен. В опыте сравниваем одни и те же лучи и учитываем все попадания.

Оценка разброса — \(J(s)=\frac1N\sum_i(q_{ix}^2+q_{iy}^2)\). Корень \(\sqrt{J(s)}\) даёт среднеквадратичный радиус пятна в метрах.

Предложите стартовое \(s\). Опишите выражение для вычисления \(J\) из массива координат попаданий. В лаборатории сначала проверьте своё выражение, затем сравните грубую и уточнённую сетки. Для ручного маршрута возьмите два луча в плоскости \(y=0\) при \(x=0{,}5\) и \(1{,}5\) м, одинаково взвесьте их и сравните \(J\) при \(s=1{,}5,\ 1{,}75,\ 2,\ 2{,}25,\ 2{,}5\) м. Уточните шаг около минимума. Проверьте найденное положение на третьем луче при \(x=1\) м. Эти два луча — ручной пример; в лаборатории весь круг апертуры будет покрыт выборкой с равными весами по площади.

**Подсказка после попытки:** у каждого луча координаты попадания линейно зависят от \(s\); можно сначала выразить эту зависимость, а затем подставлять значения из сетки.

**Самопроверка:** уточнённое значение \(J\) не больше лучшего значения на исходной сетке, если старые точки остались среди кандидатов. Пересекают ли два отражённых луча ось при одном и том же \(s\)?

In [ ]:
detector_position_m = 2.2  # Начальная проба
sphere_bundle = aperture_bundle("sphere")
hit_xy = (all_hit_xy(sphere_bundle, detector_position_m)
          if valid_detector_height(detector_position_m) else None)
# Замените None выражением для J, используя столбцы x и y массива hit_xy.
mean_squared_radius_m2 = None
report_detector_loss(detector_position_m, mean_squared_radius_m2, sphere_bundle)

**Проверьте прогноз.** По горизонтальной линии слева видно положение приёмника. Двигайте ползунок и наблюдайте, как меняются пятно и точка на кривой RMS. В статическом режиме меняйте высоту в вызове и запускайте ячейку.

In [ ]:
if valid_detector_height(detector_position_m):
    show_detector(detector_z=detector_position_m)
else:
    print("Выберите высоту приёмника от 1,5 до 2,5 м в предыдущей ячейке.")

**Уточните минимум.** Запишите оценку по графику и разрешите численный поиск, изменив False на True. Сравните грубую сетку с уточнением. Проверочная выборка плотнее и сдвинута по азимуту. Почему пятно сохраняется?

In [ ]:
run_search = False
if run_search:
    coarse_z, best_z = detector_search(sphere_bundle)
    validation_bundle = aperture_bundle("sphere", validation=True)
    print(f"Грубая сетка: {coarse_z:.3f} м · уточнение: {best_z:.3f} м")
    print(f"RMS на проверочной выборке: {100*np.sqrt(detector_mse(validation_bundle,best_z)):.3f} см")

## 6. Исправьте форму зеркала · 55–70 мин

Теперь можно заменить зеркало. Требуется собрать осевой параллельный пучок в точке на расстоянии 2 м от вершины, сохранив апертуру радиуса 1,5 м. Для осевой параболы с фокусом на высоте \(f\) используйте \(x^2=4fz\).

Вращением осевой параболы получите уравнение поверхности. Найдите глубину чаши и нарисуйте сечение на половине этой глубины. Сравните глубину с исходной сферой. Проверьте направление отражения для двух ненулевых расстояний от оси: проходит ли каждый луч через заданную точку?

**Самопроверка:** вычисленная глубина даёт требуемый радиус края; обе рассчитанные точки пересечения лежат на оси на высоте детектора с точностью округления. Осевой луч не определяет фокус: он остаётся на оси. Почему в реальном приборе пятно всё равно имеет размер?

**После расчёта.** Введите найденную высоту и угол вращения образующей от 0 до 360°. Проследите, как дуга сечения замыкается. На графике пятна масштаб указан в сантиметрах.

In [ ]:
repair_detector_m = None
turn_deg = 180
if repair_detector_m is None:
    print("○ НЕ ЗАПОЛНЕНО")
elif not valid_detector_height(repair_detector_m):
    print("△ НУЖНО ИСПРАВИТЬ: укажите высоту от 1,5 до 2,5 м.")
elif type(turn_deg) not in (float,int) or not 0 <= turn_deg <= 360:
    print("△ НУЖНО ИСПРАВИТЬ: укажите угол от 0 до 360°.")
else:
    show_paraboloid_section(turn_deg)
    show_detector("paraboloid", repair_detector_m)

## 7. Когда меньшее зеркало полезнее? · 70–80 мин

Сравните сферическое зеркало при радиусах апертуры 0,5, 1 и 1,5 м. Для каждого случая подберите положение приёмника заново. Запишите площадь \(\pi\rho^2\) и минимальный среднеквадратичный радиус. Для ручного сравнения используйте по два луча на фиксированных долях радиуса, например \(\rho/3\) и \(\rho\). Это иллюстрация тенденции, а не оценка по всей апертуре.

Сформулируйте рекомендацию для двух случаев: заказчику нужно уменьшить пятно или собрать больше света. Какие требования нужно знать для однозначного выбора?

**Самопроверка:** при изменении радиуса площадь пересчитывается отдельно от размера пятна, а положение приёмника не переносится автоматически из предыдущего опыта.

**Мост к ИИ.** В реальном опыте данные — измеренные координаты попаданий и настройки прибора. Мы выбираем параметр \(s\), минимизируем ошибку и проверяем результат на других лучах. Здесь координаты синтетические, а оптическая модель задана формулами; обученной модели нет.

**После прогноза.** Сравните площадь апертуры и остаточное пятно. Для каждого радиуса сначала ищем положение заново, затем измеряем RMS на плотной выборке.

In [ ]:
run_aperture_comparison = False
if run_aperture_comparison:
    print("Радиус, м   Площадь, м²   Высота, м   RMS, см")
    for aperture in (.5, 1., 1.5):
        _, best = detector_search(aperture_bundle("sphere",aperture))
        check = aperture_bundle("sphere",aperture,validation=True)
        print(f"{aperture:9.2f}   {np.pi*aperture**2:11.3f}   {best:9.3f}   {100*np.sqrt(detector_mse(check,best)):7.3f}")

## Выходной билет · 80–90 мин

1. Для \(z=x^2-y^2\) нарисуйте сечения \(x=0,\ y=0,\ z=0\). Почему профиль вокруг вертикальной оси не один и тот же?
2. Сравните \(x^2+y^2-z^2=1\) и \(z^2-x^2-y^2=1\) сечениями на \(z=0,\ 1,\ 2\). В каком случае между двумя кольцами есть непрерывная боковая поверхность?
3. Вернитесь к прогнозам. Закончите фразу: «Изменение уравнения изменило работу конструкции, потому что…» Приведите одно геометрическое наблюдение и одну численную проверку.

После занятия: возьмите эллипсоид с тремя разными полуосями и эллиптический параболоид с разными масштабами по \(x\) и \(y\). По двум сечениям объясните, чем каждый отличается от своего аналога вращения вокруг \(z\).

**Ваш выходной билет:** …

## Атлас сечений · после выходного билета

Сначала подставьте уравнение плоскости, затем определите множество решений. В таблице \(a,b,c,k>0\). Для эллипсоида возьмите различные \(a,b,c\), а для эллиптического параболоида — \(a\ne b\).

| Поверхность и каноническое уравнение | Горизонтальное сечение \(z=h\) | Вертикальное сечение \(y=0\) |
|---|---|---|
| Эллипсоид \(x^2/a^2+y^2/b^2+z^2/c^2=1\) | Эллипс при \(\lvert h\rvert<c\), точка при \(\lvert h\rvert=c\), затем пусто | Эллипс \(x^2/a^2+z^2/c^2=1\) |
| Однополостный гиперболоид \(x^2/a^2+y^2/b^2-z^2/c^2=1\) | Эллипс, наименьший при \(h=0\) | Гипербола \(x^2/a^2-z^2/c^2=1\) |
| Двуполостный гиперболоид \(z^2/c^2-x^2/a^2-y^2/b^2=1\) | Пусто при \(\lvert h\rvert<c\), точка на границе, затем эллипс | Гипербола \(z^2/c^2-x^2/a^2=1\) |
| Эллиптический параболоид \(z=x^2/a^2+y^2/b^2\) | Эллипс при \(h>0\), точка при \(h=0\), пусто при \(h<0\) | Парабола \(z=x^2/a^2\) |
| Гиперболический параболоид \(z=x^2/a^2-y^2/b^2\) | При \(h\ne0\) гипербола, направление ветвей зависит от знака \(h\). При \(h=0\) две пересекающиеся прямые | Парабола \(z=x^2/a^2\) |
| Эллиптический конус \(x^2/a^2+y^2/b^2=z^2/c^2\) | Эллипс при \(h\ne0\), точка при \(h=0\) | Две прямые \(x=\pm az/c\) |
| Эллиптический цилиндр \(x^2/a^2+y^2/b^2=1\) | Один и тот же эллипс на всех высотах | Две параллельные прямые \(x=\pm a\) |
| Параболический цилиндр \(y=kx^2\) | Одна и та же парабола на всех высотах | Ось \(z\). В плоскости \(y=k\) две прямые, при \(y<0\) пусто |

**Самостоятельно.** Почему одного осевого сечения недостаточно, чтобы отличить два параболоида? Подберите вторую плоскость и запишите оба уравнения сечения.

См. [страницу недели](../README.md) и [соглашения об обозначениях](../../../docs/notation.md) в репозитории.

Геометрия и результаты вычислений — учебные синтетические примеры. Контекст: [коллекция линейчатых моделей Аризонского университета](https://archive.math.arizona.edu/models/Ruled_Surfaces/index.html) и [обзор сферических зеркал OpenStax](https://openstax.org/books/university-physics-volume-3/pages/2-2-spherical-mirrors).

## Преподавателю: ориентиры и разбор

В начале соберите противоположные прогнозы. Когда пары застревают, выдавайте сначала один шаг, например подстановку плоскости, а не название поверхности. Отведите последние десять минут на индивидуальные ответы.

- **Каркас.** \(a=1,\ c=2/\sqrt3\) м; радиусы \(1,\sqrt7/2,2\) м. Из \(a=R\cos(\delta/2)\) следует \(\delta=120^\circ\); длина \(\sqrt{28}\) м. Для всех \(t\) квадрат радиуса равен \(R^2[1-4t(1-t)\sin^2(\delta/2)]\). Подстановка \(t=z/H+1/2\) возвращает то же уравнение. На концах диапазона получаются цилиндр и участок двойного конуса. Типичная ошибка: принять совпадение трёх точек за численное доказательство без объяснения роли степени многочлена.
- **Отражение.** При \(x=1\) м на сфере \(v_x=-\sqrt{15}/8,\ v_z=7/8\). Сверяйте знак и норму до обсуждения фокуса.
- **Оптимум.** Ручную выборку и полную апертуру оценивайте отдельно. Для полной апертуры при равномерном освещении ориентиры \(s\approx1{,}8935\) м, RMS \(\approx0{,}0230\) м. Для двух ручных лучей минимум можно проверить, записав \(q_{ix}(s)=a_i+s b_i\) и используя \(s=-\sum_i a_i b_i/\sum_i b_i^2\), если знаменатель ненулевой. Приближение сеткой засчитывайте с учётом шага. Частая ошибка — путать квадрат радиуса со среднеквадратичным радиусом.
- **Новое зеркало.** \(z=(x^2+y^2)/8\); глубина \(0{,}28125\) м против примерно \(0{,}29190\) м для сферы. Радиус на половине глубины равен \(1{,}5/\sqrt2\) м. Идеальный параболоид фокусирует неосевые лучи при \(z=2\) м.
- **Компромисс.** Меньшая апертура уменьшает остаточное пятно сферы, одновременно сокращая собираемую площадь. Проверяйте, что приёмник перенастроили для каждого радиуса.
- **Перенос.** У седла вертикальные сечения — параболы противоположной выпуклости; при \(z=0\) остаются \(y=\pm x\). У второго гиперболоида из билета среднее сечение пустое, при \(z=1\) оно точка, при \(z=2\) — окружность радиуса \(\sqrt3\).



### Проверка оптимума и обсуждение

Для каждого луча пусть \(b_i=v_{i,xy}/v_{iz}\), \(a_i=p_{i,xy}-p_{iz}b_i\). Тогда \(q_{i,xy}(s)=a_i+sb_i\). Раскройте сумму квадратов. При ненулевой сумме \(\sum_i\|b_i\|^2\) минимум достигается при \(s_*=-\sum_i a_i\cdot b_i/\sum_i\|b_i\|^2\). Для ограничения на интервал спроецируйте это значение на интервал. Если все \(b_i=0\), ошибка постоянна.

Ориентиры при повторной настройке для апертур 0,5, 1 и 1,5 м: RMS около 0,067, 0,582 и 2,297 см соответственно. Им соответствуют площади около 0,785, 3,142 и 7,069 м². Для оценки выхода попросите связать вид сечения со знаком правой части, а размер пятна — с тем, что лучи на сфере пересекают ось на разных высотах.


In [ ]:
points, directions = aperture_bundle("sphere")
slopes = directions[:,:2] / directions[:,2,None]
intercepts = points[:,:2] - points[:,2,None]*slopes
denominator = np.sum(slopes**2)
if denominator > 0:
    analytic = np.clip(-np.sum(intercepts*slopes)/denominator, *_DETECTOR_BOUNDS)
    _, numerical = detector_search((points,directions))
    assert abs(analytic-numerical) <= 0.00051
    print(f"Аналитический оптимум: {analytic:.5f} м")
else:
    print("Ошибка не зависит от высоты.")
assert detector_mse(aperture_bundle("paraboloid",validation=True),TARGET_FOCUS_M) < 1e-27